# Machine Learning Project
# Cropping Cards for training use
# Kylle Waldie

### Imports

In [1]:
import cv2
import os

## Cropping cards

In [2]:
def crop_card(image_path, save_path=None, debug=False):
    img = cv2.imread(image_path)
    if img is None:
        return None

    h_img, w_img = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 50, 180)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    slab = None
    slab_area = 0

    # Step 1: find slab
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        area = w * h
        aspect = h / float(w)

        if area > slab_area and aspect > 1.3 and area > 0.25 * w_img * h_img:
            slab = (x, y, w, h)
            slab_area = area

    # Step 2: crop card within slab
    if slab:
        x, y, w, h = slab

        # Leave safe margins (reduce clipping)
        top_margin = int(h * 0.16)       # 16% off top
        bottom_margin = int(h * 0.05)    # 5% at bottom
        side_margin = int(w * 0.04)      # 4% on each side

        card_x = x + side_margin
        card_y = y + top_margin
        card_w = w - 2 * side_margin
        card_h = h - top_margin - bottom_margin

        # Clamp to image
        card_x = max(0, card_x)
        card_y = max(0, card_y)
        card_w = min(card_w, w_img - card_x)
        card_h = min(card_h, h_img - card_y)

        cropped = img[card_y:card_y+card_h, card_x:card_x+card_w]
    else:
        # fallback: loose center crop
        margin = int(min(w_img, h_img) * 0.08)
        cropped = img[margin:h_img-margin, margin:w_img-margin]

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, cropped)

    if debug:
        cv2.imshow("crop", cropped)
        cv2.waitKey(300)

    return cropped

### Batch Crop

In [3]:
def batch_crop_images(input_folder="dataset/raw", output_folder="dataset/cropped"):
    for grade_folder in os.listdir(input_folder):
        grade_path = os.path.join(input_folder, grade_folder)
        if not os.path.isdir(grade_path):
            continue
        
        save_grade_path = os.path.join(output_folder, grade_folder)
        os.makedirs(save_grade_path, exist_ok=True)
        
        for img_file in os.listdir(grade_path):
            img_path = os.path.join(grade_path, img_file)
            save_path = os.path.join(save_grade_path, img_file)
            
            crop_card(img_path, save_path=save_path)

### Running cropping

In [4]:
batch_crop_images(input_folder="dataset/raw", output_folder="dataset/cropped")